# Flow Matching on N x N Mazes

Compare velocity MSE and endpoint-residual MSE across 10 paired weight initializations. Benchmark the original Gaussian-prior models with ODE and stochastic sampling, then train 20 additional models from a fixed all-black source and learn a noisy bridge and test its matched stochastic sampler from that singular start. Evaluate 32x32 and 64x64 mazes with 100 sampling steps.

Each replicate pair shares its initial weights and training examples, so the comparison can show both initialization variability and differences between objectives. Solvability is measured as the fraction of generated mazes whose entrance connects to the exit.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))

import copy
import importlib
import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import clear_output, display, Markdown
import ipywidgets as widgets
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from plotly.subplots import make_subplots

import _shared.maze_data as maze_data_module

importlib.reload(maze_data_module)
from _shared.maze_data import generate_dataset, solvability_rate
import _shared.model as model_module

importlib.reload(model_module)
from _shared.model import (
    Denoiser,
    decode_probabilities,
    embed_binary,
    make_binary_embeddings,
)

SEED = 7
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Synthetic Dataset

Randomized DFS generates perfect mazes. `0` means wall; `1` means passage.

In [2]:
MAZE_SIZE = 32  # Training resolution; evaluation also checks 64x64 below.
N_MAZES = 12_000
VAL_FRACTION = 0.1
mazes = generate_dataset(N_MAZES, seed=SEED, size=MAZE_SIZE)
n_val = int(N_MAZES * VAL_FRACTION)
train_mazes = torch.from_numpy(mazes[:-n_val].copy()).long()
val_mazes = torch.from_numpy(mazes[-n_val:].copy()).long()

fig = go.Figure(
    go.Heatmap(
        z=mazes[0],
        zmin=0,
        zmax=1,
        colorscale=[[0, "black"], [1, "white"]],
        showscale=False,
    )
)
fig.update_layout(
    title=f"Synthetic {MAZE_SIZE}x{MAZE_SIZE} maze example",
    width=420,
    height=420,
    yaxis=dict(autorange="reversed", scaleanchor="x"),
)
fig.show()

## 2. Forward Flow Path

For a clean embedded maze `x_data` and independent Gaussian noise `z`, define the conditional path

`x_t = (1 - t) * z + t * x_data`, with `t` sampled uniformly from `[0, 1]`.

Its target velocity is constant along the path: `v_target = x_data - z`. This panel decodes the continuous states using the same fixed-temperature embedding dot-product visualization at every time; the visualization does not alter the path.

In [3]:
EMBEDDING_DIM = 2
TEMPERATURE = 0.5
binary_embeddings = make_binary_embeddings(EMBEDDING_DIM, DEVICE)

preview_maze = torch.from_numpy(mazes[0].copy()).long().unsqueeze(0).to(DEVICE)
x_data = embed_binary(preview_maze, binary_embeddings)
preview_noise = torch.randn_like(x_data)
preview_times = [0.0, 0.1, 0.25, 0.4, 0.5, 0.6, 0.75, 0.9, 1.0]
with torch.no_grad():
    preview_frames = []
    for t in preview_times:
        t_batch = torch.full((1,), t, device=DEVICE)
        x_t = (1.0 - t_batch[:, None, None, None]) * preview_noise + t_batch[
            :, None, None, None
        ] * x_data
        probabilities = decode_probabilities(x_t, binary_embeddings, TEMPERATURE)
        preview_frames.append(probabilities[0, 1].cpu().numpy())

path_fig = make_subplots(
    rows=1,
    cols=len(preview_times),
    subplot_titles=[f"t={t:g}" for t in preview_times],
)
for col, frame in enumerate(preview_frames, start=1):
    path_fig.add_trace(
        go.Heatmap(
            z=frame,
            zmin=0,
            zmax=1,
            colorscale=[[0, "black"], [1, "white"]],
            showscale=False,
        ),
        row=1,
        col=col,
    )
path_fig.update_layout(
    title="Linear flow path: black = p(0), white = p(1)",
    width=1700,
    height=360,
)
path_fig.update_xaxes(showticklabels=False)
path_fig.update_yaxes(showticklabels=False, autorange="reversed")
path_fig.show()

## 3. Model Architecture

The shared U-Net is fully convolutional and accepts input shape `(B, 2, N, N)` for the tested square resolutions. It has base channels `64`, encoder widths `64 -> 128 -> 256`, residual blocks conditioned by a `128`-dimensional time embedding, and a two-channel velocity output. Time is scaled to `[0, 100]` before sinusoidal embedding. See `notebooks/_shared/model.py` for the implementation.


In [4]:
BASE_CHANNELS = 64
TIME_DIM = 128
TIME_EMBEDDING_SCALE = 100.0
N_REPLICATES = 10
OBJECTIVES = ("velocity", "endpoint_residual")
OBJECTIVE_LABELS = {
    "velocity": "Velocity MSE",
    "endpoint_residual": "Endpoint-residual MSE",
}
OBJECTIVE_COLORS = {
    "velocity": "#4263eb",
    "endpoint_residual": "#e76f00",
}

# Pair the two objectives by initialization seed; each pair starts identically.
models = {}
MODEL_META = {}
for replicate in range(N_REPLICATES):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(SEED + 1000 + replicate)
        initial_model = Denoiser(
            embedding_dim=EMBEDDING_DIM,
            base_channels=BASE_CHANNELS,
            time_dim=TIME_DIM,
        )
    for objective in OBJECTIVES:
        key = f"{objective}_{replicate + 1:02d}"
        candidate = (
            initial_model if objective == "velocity" else copy.deepcopy(initial_model)
        )
        models[key] = candidate.to(DEVICE)
        MODEL_META[key] = {"objective": objective, "replicate": replicate + 1}

## 4. Compare two flow-matching objectives across initializations

Train 10 paired replicates for each objective. Within each pair, both models start from identical weights and see the same maze, noise, and time batches.

1. **Velocity MSE:** $\operatorname{MSE}(v_\theta, x_1-z)$.
2. **Endpoint-residual MSE:** $\operatorname{MSE}((1-t_0)v_\theta, x_1-x_{t_0})$.

Both models predict the path velocity. The second loss weights velocity errors by $(1-t_0)^2$. The training plot reports mean train and validation losses across the 10 replicates for each objective.


In [5]:
BATCH_SIZE = 128
EPOCHS = 30
LEARNING_RATE = 2e-4
PLOT_EVERY = 1

train_loader = DataLoader(
    TensorDataset(train_mazes), batch_size=BATCH_SIZE, shuffle=True, drop_last=True
)
val_loader = DataLoader(TensorDataset(val_mazes), batch_size=BATCH_SIZE, shuffle=False)
optimizers = {
    name: torch.optim.AdamW(candidate.parameters(), lr=LEARNING_RATE)
    for name, candidate in models.items()
}


def sample_training_context(maze_batch, generator=None):
    x_data = embed_binary(maze_batch.to(DEVICE), binary_embeddings)
    noise = torch.randn(
        x_data.shape, generator=generator, device=DEVICE, dtype=x_data.dtype
    )
    t0 = (1.0 - 1e-3) * torch.rand(
        x_data.shape[0], generator=generator, device=DEVICE, dtype=x_data.dtype
    )
    t0_image = t0[:, None, None, None]
    x_t0 = (1.0 - t0_image) * noise + t0_image * x_data
    return x_data, noise, t0, x_t0


def objective_loss(context, candidate, objective):
    x_data, noise, t0, x_t0 = context
    predicted_velocity = candidate(x_t0, t0 * TIME_EMBEDDING_SCALE)
    if objective == "velocity":
        return F.mse_loss(predicted_velocity, x_data - noise)
    endpoint_residual = x_data - x_t0
    scale = (1.0 - t0)[:, None, None, None]
    return F.mse_loss(scale * predicted_velocity, endpoint_residual)


@torch.no_grad()
def validation_losses():
    generator = torch.Generator(device=DEVICE).manual_seed(SEED + 401)
    was_training = {name: candidate.training for name, candidate in models.items()}
    for candidate in models.values():
        candidate.eval()
    totals = {name: [] for name in models}
    try:
        for (maze_batch,) in val_loader:
            context = sample_training_context(maze_batch, generator)
            for name, candidate in models.items():
                objective = MODEL_META[name]["objective"]
                totals[name].append(
                    objective_loss(context, candidate, objective).item()
                )
        return {name: float(np.mean(values)) for name, values in totals.items()}
    finally:
        for name, candidate in models.items():
            candidate.train(was_training[name])


train_history = {objective: [] for objective in OBJECTIVES}
val_history = {objective: [] for objective in OBJECTIVES}
loss_fig = go.Figure()
for objective in OBJECTIVES:
    color = OBJECTIVE_COLORS[objective]
    loss_fig.add_scatter(
        name=f"{OBJECTIVE_LABELS[objective]} train",
        mode="lines",
        line=dict(color=color, width=2),
    )
    loss_fig.add_scatter(
        name=f"{OBJECTIVE_LABELS[objective]} validation",
        mode="lines",
        line=dict(color=color, width=2, dash="dot"),
    )
loss_fig.update_layout(
    title="Mean loss across 10 paired initializations",
    xaxis_title="Epoch",
    yaxis_title="Objective MSE (log scale; scales differ)",
    yaxis_type="log",
    template="plotly_white",
    width=1050,
    height=520,
    legend=dict(orientation="h", y=-0.22),
)

epoch_progress = widgets.IntProgress(
    value=0,
    min=0,
    max=EPOCHS * len(train_loader) * len(models),
    description="Training",
    bar_style="info",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%"),
)
display(epoch_progress)
display(loss_fig)
for epoch in range(1, EPOCHS + 1):
    for candidate in models.values():
        candidate.train()
    epoch_values = {name: [] for name in models}
    for batch_index, (maze_batch,) in enumerate(train_loader, start=1):
        context = sample_training_context(maze_batch)
        for model_index, (name, candidate) in enumerate(models.items(), start=1):
            objective = MODEL_META[name]["objective"]
            loss = objective_loss(context, candidate, objective)
            optimizers[name].zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(candidate.parameters(), max_norm=1.0)
            optimizers[name].step()
            epoch_values[name].append(loss.item())
            epoch_progress.value = (
                (epoch - 1) * len(train_loader) * len(models)
                + (batch_index - 1) * len(models)
                + model_index
            )

    for objective in OBJECTIVES:
        run_means = [
            float(np.mean(epoch_values[name]))
            for name in models
            if MODEL_META[name]["objective"] == objective
        ]
        train_history[objective].append(float(np.mean(run_means)))
    validation_by_model = validation_losses()
    for objective in OBJECTIVES:
        run_values = [
            validation_by_model[name]
            for name in models
            if MODEL_META[name]["objective"] == objective
        ]
        val_history[objective].append(float(np.mean(run_values)))

    if epoch % PLOT_EVERY == 0 or epoch == 1:
        for objective_index, objective in enumerate(OBJECTIVES):
            train_trace = 2 * objective_index
            val_trace = train_trace + 1
            loss_fig.data[train_trace].x = list(range(1, epoch + 1))
            loss_fig.data[train_trace].y = np.maximum(train_history[objective], 1e-12)
            loss_fig.data[val_trace].x = list(range(1, epoch + 1))
            loss_fig.data[val_trace].y = np.maximum(val_history[objective], 1e-12)
        loss_fig.update_layout(
            title=f"Mean loss across 10 paired initializations | epoch {epoch}/{EPOCHS}"
        )
        clear_output(wait=True)
        display(epoch_progress)
        display(loss_fig)

IntProgress(value=50400, bar_style='info', description='Training', layout=Layout(width='100%'), max=50400, sty…

## 4.1 Train a stochastic bridge from an all-black start

Train 10 paired initializations for each objective. Every trajectory starts at the same black one-hot grid $b$ and ends at a sampled maze $x_1$. Randomness is introduced along the path, rather than by randomizing the source.

### Training states and SDE drift

For each batch item, independently sample a maze, $t\in[\delta,1-\delta]$ and a full-grid Gaussian tensor $\eta\sim\mathcal N(0,I)$. With `SINGULAR_BRIDGE_STRENGTH` $\lambda_b>0$ and $\delta=10^{-3}$, form

$$
m_t=(1-t)b+t x_1,\qquad
n_t=\sqrt{2\lambda_b t(1-t)}\,\eta,\qquad
x_t=m_t+n_t.
$$

Thus $x_0=b$ exactly, intermediate states contain noise, and $x_1$ is the clean maze. These marginals admit a conditional SDE with decreasing diffusion:

$$
dx_t=a^*(x_t,t;x_1)\,dt+\sqrt{2\lambda_b(1-t)}\,dW_t,
\qquad
a^*=x_1-b-\frac{n_t}{2(1-t)}.
$$

The variance satisfies $q'(t)=-q(t)/(1-t)+2\lambda_b(1-t)$, giving $q(t)=2\lambda_b t(1-t)$. The network receives only $(x_t,t)$ and learns the conditional mean of this drift. Its architecture is unchanged; for the black-start models its output represents **SDE drift**, rather than the velocity of the Gaussian linear path.

### Two losses for the same drift

**Velocity / drift MSE:** compare $a_\theta(x_t,t)$ directly with $a^*$.

**Endpoint-residual MSE:** infer the endpoint from the predicted drift,

$$
\widehat{x}_1=b+\frac{2(1-t)a_\theta+(x_t-b)}{2-t},
$$

then minimize $\operatorname{MSE}(\widehat{x}_1-x_t,\;x_1-x_t)$. Equivalently, this is drift MSE weighted by $[2(1-t)/(2-t)]^2$. Both objectives therefore learn the same drift with different weighting across time; the deterministic-path residual formula is not used for this bridge.

The noisy interpolant construction follows [Stochastic Interpolants](https://arxiv.org/abs/2303.08797). The decreasing diffusion and the drift above are the choices used here. The 20 models are saved to a new bridge checkpoint with its noise strength and path version. Previous deterministic black-start checkpoints require retraining.

Run the setup cell below for training or loading. To reuse a checkpoint, skip **Train singular bridge models** and **Save singular bridge models**, then run **Load singular bridge models** before Section 6.


In [6]:
SINGULAR_REPLICATES = 10
SINGULAR_EPOCHS = 30
SINGULAR_BATCH_SIZE = 128
SINGULAR_LEARNING_RATE = 2e-4
ARTIFACTS = PROJECT_ROOT / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
SINGULAR_BRIDGE_STRENGTH = 1.0
SINGULAR_TIME_MARGIN = 1e-3
SINGULAR_PATH_VERSION = "black_stochastic_bridge_v1"
SINGULAR_CHECKPOINT_PATH = (
    ARTIFACTS / "maze_flow_matching_singular_bridge_replicates.pt"
)
if not np.isfinite(SINGULAR_BRIDGE_STRENGTH) or SINGULAR_BRIDGE_STRENGTH <= 0:
    raise ValueError("SINGULAR_BRIDGE_STRENGTH must be finite and positive")
if not 0 < SINGULAR_TIME_MARGIN < 0.5:
    raise ValueError("SINGULAR_TIME_MARGIN must lie between 0 and 0.5")

singular_models = {}
singular_model_meta = {}
for replicate in range(SINGULAR_REPLICATES):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(SEED + 2000 + replicate)
        initial_model = Denoiser(
            embedding_dim=EMBEDDING_DIM,
            base_channels=BASE_CHANNELS,
            time_dim=TIME_DIM,
        )
    for objective in OBJECTIVES:
        key = f"singular_{objective}_{replicate + 1:02d}"
        candidate = (
            initial_model if objective == "velocity" else copy.deepcopy(initial_model)
        )
        singular_models[key] = candidate.to(DEVICE)
        singular_model_meta[key] = {
            "objective": objective,
            "replicate": replicate + 1,
            "initialization": "singular_black",
            "path_version": SINGULAR_PATH_VERSION,
            "bridge_strength": SINGULAR_BRIDGE_STRENGTH,
            "prediction": "sde_drift",
        }


def singular_endpoint_from_drift(state, drift, time, black):
    # time is a scalar or a [B, 1, 1, 1] tensor.
    remaining = 1.0 - time
    return black + (2.0 * remaining * drift + state - black) / (2.0 - time)


def sample_singular_context(maze_batch, generator=None):
    x_data = embed_binary(maze_batch.to(DEVICE), binary_embeddings)
    z_black = binary_embeddings[0].view(1, EMBEDDING_DIM, 1, 1).expand_as(x_data)
    t0 = SINGULAR_TIME_MARGIN + (1.0 - 2.0 * SINGULAR_TIME_MARGIN) * torch.rand(
        x_data.shape[0], generator=generator, device=DEVICE, dtype=x_data.dtype
    )
    t_image = t0[:, None, None, None]
    epsilon = torch.randn(
        x_data.shape, generator=generator, device=DEVICE, dtype=x_data.dtype
    )
    bridge_noise = (
        2.0 * SINGULAR_BRIDGE_STRENGTH * t_image * (1.0 - t_image)
    ).sqrt() * epsilon
    x_t0 = (1.0 - t_image) * z_black + t_image * x_data + bridge_noise
    target_drift = x_data - z_black - bridge_noise / (2.0 * (1.0 - t_image))
    return x_data, z_black, t0, x_t0, target_drift


def singular_objective_loss(context, candidate, objective):
    x_data, z_black, t0, x_t0, target_drift = context
    predicted_drift = candidate(x_t0, t0 * TIME_EMBEDDING_SCALE)
    if objective == "velocity":
        return F.mse_loss(predicted_drift, target_drift)
    if objective != "endpoint_residual":
        raise ValueError(f"Unknown bridge objective: {objective}")
    predicted_endpoint = singular_endpoint_from_drift(
        x_t0, predicted_drift, t0[:, None, None, None], z_black
    )
    return F.mse_loss(predicted_endpoint - x_t0, x_data - x_t0)


for name, candidate in singular_models.items():
    candidate.singular_bridge_strength = SINGULAR_BRIDGE_STRENGTH
    candidate.singular_path_version = SINGULAR_PATH_VERSION

for name in models:
    MODEL_META[name].setdefault("initialization", "gaussian")
models.update(singular_models)
MODEL_META.update(singular_model_meta)

### Train singular bridge models

Skip this cell when loading a trained bridge checkpoint below.


In [7]:
singular_train_loader = DataLoader(
    TensorDataset(train_mazes),
    batch_size=SINGULAR_BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)
singular_val_loader = DataLoader(
    TensorDataset(val_mazes), batch_size=SINGULAR_BATCH_SIZE, shuffle=False
)

singular_optimizers = {
    name: torch.optim.AdamW(candidate.parameters(), lr=SINGULAR_LEARNING_RATE)
    for name, candidate in singular_models.items()
}


@torch.no_grad()
def singular_validation_losses():
    generator = torch.Generator(device=DEVICE).manual_seed(SEED + 2401)
    was_training = {
        name: candidate.training for name, candidate in singular_models.items()
    }
    for candidate in singular_models.values():
        candidate.eval()
    totals = {name: [] for name in singular_models}
    try:
        for (maze_batch,) in singular_val_loader:
            context = sample_singular_context(maze_batch, generator)
            for name, candidate in singular_models.items():
                totals[name].append(
                    singular_objective_loss(
                        context, candidate, singular_model_meta[name]["objective"]
                    ).item()
                )
        return {name: float(np.mean(values)) for name, values in totals.items()}
    finally:
        for name, candidate in singular_models.items():
            candidate.train(was_training[name])


singular_train_history = {objective: [] for objective in OBJECTIVES}
singular_val_history = {objective: [] for objective in OBJECTIVES}
singular_loss_labels = {
    "velocity": "Drift MSE",
    "endpoint_residual": "Endpoint-residual MSE",
}
singular_loss_fig = go.Figure()
for objective in OBJECTIVES:
    color = OBJECTIVE_COLORS[objective]
    singular_loss_fig.add_scatter(
        name=f"{singular_loss_labels[objective]} train",
        mode="lines",
        line=dict(color=color, width=2),
    )
    singular_loss_fig.add_scatter(
        name=f"{singular_loss_labels[objective]} validation",
        mode="lines",
        line=dict(color=color, width=2, dash="dot"),
    )
singular_loss_fig.update_layout(
    title="Black-start stochastic bridge training | mean loss across 10 paired initializations",
    xaxis_title="Epoch",
    yaxis_title="Objective MSE (log scale; scales differ)",
    yaxis_type="log",
    template="plotly_white",
    autosize=True,
    height=480,
    legend=dict(orientation="h", y=-0.22),
)
singular_progress = widgets.IntProgress(
    value=0,
    min=0,
    max=SINGULAR_EPOCHS * len(singular_train_loader) * len(singular_models),
    description="Singular-start training",
    bar_style="info",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%"),
)
display(singular_progress)
display(singular_loss_fig)
for epoch in range(1, SINGULAR_EPOCHS + 1):
    for candidate in singular_models.values():
        candidate.train()
    epoch_values = {name: [] for name in singular_models}
    for batch_index, (maze_batch,) in enumerate(singular_train_loader, start=1):
        context = sample_singular_context(maze_batch)
        for model_index, (name, candidate) in enumerate(
            singular_models.items(), start=1
        ):
            objective = singular_model_meta[name]["objective"]
            loss = singular_objective_loss(context, candidate, objective)
            singular_optimizers[name].zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(candidate.parameters(), max_norm=1.0)
            singular_optimizers[name].step()
            epoch_values[name].append(loss.item())
            singular_progress.value = (
                (epoch - 1) * len(singular_train_loader) * len(singular_models)
                + (batch_index - 1) * len(singular_models)
                + model_index
            )
    for objective in OBJECTIVES:
        train_values = [
            float(np.mean(epoch_values[name]))
            for name in singular_models
            if singular_model_meta[name]["objective"] == objective
        ]
        singular_train_history[objective].append(float(np.mean(train_values)))
    val_by_model = singular_validation_losses()
    for objective in OBJECTIVES:
        val_values = [
            val_by_model[name]
            for name in singular_models
            if singular_model_meta[name]["objective"] == objective
        ]
        singular_val_history[objective].append(float(np.mean(val_values)))
    for objective_index, objective in enumerate(OBJECTIVES):
        train_trace = 2 * objective_index
        singular_loss_fig.data[train_trace].x = list(range(1, epoch + 1))
        singular_loss_fig.data[train_trace].y = np.maximum(
            singular_train_history[objective], 1e-12
        )
        singular_loss_fig.data[train_trace + 1].x = list(range(1, epoch + 1))
        singular_loss_fig.data[train_trace + 1].y = np.maximum(
            singular_val_history[objective], 1e-12
        )
    singular_loss_fig.update_layout(
        title=f"Black-start stochastic bridge training | epoch {epoch}/{SINGULAR_EPOCHS}"
    )
    clear_output(wait=True)
    display(singular_progress)
    display(singular_loss_fig)
singular_progress.bar_style = "success"

IntProgress(value=50400, bar_style='info', description='Singular-start training', layout=Layout(width='100%'),…

## 5. Save and load trained models

### Save Gaussian-start models

Save the Gaussian velocity and endpoint-residual replicates, pixel embeddings and model configuration. Singular models have their own checkpoint below.


In [ ]:
ARTIFACTS = PROJECT_ROOT / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
checkpoint_path = ARTIFACTS / "maze_flow_matching_replicates.pt"
torch.save(
    {
        "models_state_dict": {
            name: candidate.state_dict()
            for name, candidate in models.items()
            if MODEL_META[name].get("initialization", "gaussian") == "gaussian"
        },
        "model_meta": {
            name: metadata
            for name, metadata in MODEL_META.items()
            if metadata.get("initialization", "gaussian") == "gaussian"
        },
        "binary_embeddings": binary_embeddings.detach().cpu(),
        "config": {
            "embedding_dim": EMBEDDING_DIM,
            "base_channels": BASE_CHANNELS,
            "time_dim": TIME_DIM,
            "time_embedding_scale": TIME_EMBEDDING_SCALE,
            "n_replicates_per_objective": N_REPLICATES,
            "training_maze_size": MAZE_SIZE,
            "objectives": list(OBJECTIVES),
            "epochs": EPOCHS,
        },
    },
    checkpoint_path,
)

### Load Gaussian-start models

Run Sections 1–3 and the setup cell in Section 4.1, then this cell to skip Gaussian training. It reconstructs the saved models and preserves the singular model collection.


In [5]:
checkpoint_path = PROJECT_ROOT / "artifacts" / "maze_flow_matching_replicates.pt"
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
gaussian_config = checkpoint["config"]
loaded_embeddings = checkpoint["binary_embeddings"].to(DEVICE)
if any(meta.get("initialization") == "singular_black" for meta in MODEL_META.values()):
    if not torch.equal(loaded_embeddings, binary_embeddings):
        raise ValueError("Gaussian and singular model embeddings must match")
binary_embeddings = loaded_embeddings
EMBEDDING_DIM = int(gaussian_config["embedding_dim"])
BASE_CHANNELS = int(gaussian_config["base_channels"])
TIME_DIM = int(gaussian_config["time_dim"])
TIME_EMBEDDING_SCALE = float(gaussian_config["time_embedding_scale"])
N_REPLICATES = int(gaussian_config["n_replicates_per_objective"])
loaded_gaussian_models = {}
loaded_gaussian_meta = {}
for name, metadata in checkpoint["model_meta"].items():
    if metadata.get("initialization", "gaussian") != "gaussian":
        continue
    candidate = models.get(name)
    if (
        candidate is None
        or candidate.input.in_channels != EMBEDDING_DIM
        or candidate.input.out_channels != BASE_CHANNELS
        or candidate.time_dim != TIME_DIM
    ):
        candidate = Denoiser(
            embedding_dim=EMBEDDING_DIM, base_channels=BASE_CHANNELS, time_dim=TIME_DIM
        ).to(DEVICE)
    candidate.load_state_dict(checkpoint["models_state_dict"][name])
    candidate.eval()
    loaded_gaussian_models[name] = candidate
    loaded_gaussian_meta[name] = dict(metadata, initialization="gaussian")
for name in list(models):
    if MODEL_META[name].get("initialization", "gaussian") == "gaussian":
        del models[name]
        del MODEL_META[name]
models.update(loaded_gaussian_models)
MODEL_META.update(loaded_gaussian_meta)

### Save singular bridge models

Run after training in Section 4.1. Save all singular replicates, embeddings, architecture and the matched bridge settings to `artifacts/maze_flow_matching_singular_bridge_replicates.pt`. Skip when reusing a saved checkpoint.


In [8]:
SINGULAR_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "models_state_dict": {
            name: model.state_dict() for name, model in singular_models.items()
        },
        "model_meta": singular_model_meta,
        "binary_embeddings": binary_embeddings.detach().cpu(),
        "config": {
            "embedding_dim": EMBEDDING_DIM,
            "base_channels": BASE_CHANNELS,
            "time_dim": TIME_DIM,
            "time_embedding_scale": TIME_EMBEDDING_SCALE,
            "training_maze_size": MAZE_SIZE,
            "objectives": list(OBJECTIVES),
            "n_replicates_per_objective": SINGULAR_REPLICATES,
            "epochs": SINGULAR_EPOCHS,
            "source_state": "fixed black one-hot embedding",
            "path_version": SINGULAR_PATH_VERSION,
            "bridge_strength": SINGULAR_BRIDGE_STRENGTH,
            "time_margin": SINGULAR_TIME_MARGIN,
            "prediction": "sde_drift",
        },
    },
    SINGULAR_CHECKPOINT_PATH,
)

### Load singular bridge models

Run after the Section 4.1 setup cell to skip singular training. Restore weights, bridge strength, path version and model metadata; add the loaded models to the shared collection used by all benchmarks.


In [ ]:
SINGULAR_CHECKPOINT_PATH = (
    PROJECT_ROOT / "artifacts" / "maze_flow_matching_singular_bridge_replicates.pt"
)
singular_checkpoint = torch.load(
    SINGULAR_CHECKPOINT_PATH, map_location=DEVICE, weights_only=True
)
bridge_config = singular_checkpoint["config"]
if bridge_config.get("path_version") != SINGULAR_PATH_VERSION:
    raise ValueError(
        "This checkpoint uses an incompatible singular path; retrain the bridge models"
    )
if not torch.equal(
    singular_checkpoint["binary_embeddings"].to(binary_embeddings), binary_embeddings
):
    raise ValueError(
        "Bridge checkpoint embeddings differ from the current dataset embeddings"
    )
if (
    bridge_config.get("time_embedding_scale", TIME_EMBEDDING_SCALE)
    != TIME_EMBEDDING_SCALE
):
    raise ValueError("Bridge checkpoint time scale differs from the Gaussian models")
SINGULAR_BRIDGE_STRENGTH = float(bridge_config["bridge_strength"])
SINGULAR_TIME_MARGIN = float(bridge_config["time_margin"])
SINGULAR_REPLICATES = int(bridge_config["n_replicates_per_objective"])
if not np.isfinite(SINGULAR_BRIDGE_STRENGTH) or SINGULAR_BRIDGE_STRENGTH <= 0:
    raise ValueError("Checkpoint bridge strength must be finite and positive")
if not 0 < SINGULAR_TIME_MARGIN < 0.5:
    raise ValueError("Checkpoint time margin must lie between 0 and 0.5")

singular_model_meta = singular_checkpoint["model_meta"]
loaded_singular_models = {}
for name, metadata in singular_model_meta.items():
    candidate = models.get(name)
    if (
        candidate is None
        or candidate.input.in_channels
        != bridge_config.get("embedding_dim", EMBEDDING_DIM)
        or candidate.input.out_channels
        != bridge_config.get("base_channels", BASE_CHANNELS)
        or candidate.time_dim != bridge_config.get("time_dim", TIME_DIM)
    ):
        candidate = Denoiser(
            embedding_dim=bridge_config.get("embedding_dim", EMBEDDING_DIM),
            base_channels=bridge_config.get("base_channels", BASE_CHANNELS),
            time_dim=bridge_config.get("time_dim", TIME_DIM),
        ).to(DEVICE)
    candidate.load_state_dict(singular_checkpoint["models_state_dict"][name])
    candidate.singular_bridge_strength = SINGULAR_BRIDGE_STRENGTH
    candidate.singular_path_version = SINGULAR_PATH_VERSION
    candidate.eval()
    loaded_singular_models[name] = candidate

for name in list(models):
    if MODEL_META[name].get("initialization") == "singular_black":
        del models[name]
        del MODEL_META[name]
singular_models = loaded_singular_models
models.update(singular_models)
MODEL_META.update(singular_model_meta)

## 6. Generated examples and benchmarks: initialization × objective × sampler

Compare six combinations: Gaussian start with velocity or endpoint-residual training, each sampled by ODE or stochastic SDE; and singular black start with the two stochastic-bridge objectives. Every run uses 100 steps. Evaluate 32×32 and 64×64 mazes with shared Gaussian starts or the exact black grid, and common seeded Gaussian increments across model replicates.

### Gaussian-start sampling

For $x_t=(1-t)z+t x_1$ with Gaussian $z$, use the score-corrected SDE:

$$
x_{t+\Delta t}=x_t+\Delta t\left[(1+\lambda t)v_\theta(x_t,t)-\lambda x_t\right]+\sqrt{2\lambda(1-t)\Delta t}\,\epsilon.
$$

`STOCHASTIC_STRENGTH` controls $\lambda$ for these Gaussian models; zero recovers ODE sampling.

### Matched black-start sampling

Use the drift and $\lambda_b$ learned in Section 4.1, with fresh Gaussian noise at every step. At time $t$, infer $\widehat{x}_1$ from the network drift, freeze that endpoint during the step, and let $u=t+\Delta t$:

$$
\widehat m_s=(1-s)b+s\widehat{x}_1,\qquad
x_u=\widehat m_u+\sqrt{\frac{1-u}{1-t}}(x_t-\widehat m_t)
+\sqrt{2\lambda_b(1-u)\Delta t}\,\epsilon.
$$

This is the exact conditional transition for a fixed known endpoint; freezing a learned endpoint gives a finite-step approximation to the learned SDE. It has the learned drift in the small-step limit. The first step introduces the bridge variance $2\lambda_b\Delta t(1-\Delta t)$; the final step removes the bridge noise and returns $\widehat{x}_1$. The bridge noise strength comes from its training checkpoint and is independent of the Gaussian sampler setting.

Cache generated mazes for all later metrics and show one final sample per trained model. The following cell shows a six-frame trajectory for each combination.


In [9]:
N_METRIC_SAMPLES = 128
GENERATION_STEPS = 100
ODE_STEPS = GENERATION_STEPS
STOCHASTIC_STRENGTH = 1.0
METRIC_BATCH_SIZE = 16
EVALUATION_MAZE_SIZES = tuple(dict.fromkeys((MAZE_SIZE, 64)))
SAMPLERS = ("ode", "stochastic")
BENCHMARK_GROUPS = (
    ("gaussian", "velocity", "ode"),
    ("gaussian", "velocity", "stochastic"),
    ("gaussian", "endpoint_residual", "ode"),
    ("gaussian", "endpoint_residual", "stochastic"),
    ("singular_black", "velocity", "stochastic"),
    ("singular_black", "endpoint_residual", "stochastic"),
)
BENCHMARK_LABELS = {
    ("gaussian", "velocity", "ode"): "Gaussian | Velocity | ODE",
    ("gaussian", "velocity", "stochastic"): "Gaussian | Velocity | Stochastic",
    ("gaussian", "endpoint_residual", "ode"): "Gaussian | Endpoint | ODE",
    ("gaussian", "endpoint_residual", "stochastic"): "Gaussian | Endpoint | Stochastic",
    ("singular_black", "velocity", "stochastic"): "Black bridge | Drift | Stochastic",
    (
        "singular_black",
        "endpoint_residual",
        "stochastic",
    ): "Black bridge | Endpoint | Stochastic",
}
BENCHMARK_TICKS = {
    group: label.replace(" | ", "<br>") for group, label in BENCHMARK_LABELS.items()
}
BENCHMARK_COLORS = {
    ("gaussian", "velocity", "ode"): "#4263eb",
    ("gaussian", "velocity", "stochastic"): "#0891b2",
    ("gaussian", "endpoint_residual", "ode"): "#e76f00",
    ("gaussian", "endpoint_residual", "stochastic"): "#c02675",
    ("singular_black", "velocity", "stochastic"): "#159447",
    ("singular_black", "endpoint_residual", "stochastic"): "#8b5cf6",
}


@torch.no_grad()
def sample_flow(
    candidate,
    initial_state,
    steps=GENERATION_STEPS,
    sampler="ode",
    stochastic_strength=STOCHASTIC_STRENGTH,
    initialization="gaussian",
    generator=None,
    capture_steps=None,
):
    if steps < 1 or int(steps) != steps:
        raise ValueError("steps must be a positive integer")
    if sampler not in SAMPLERS:
        raise ValueError(f"Unknown sampler: {sampler}")
    if initialization not in ("gaussian", "singular_black"):
        raise ValueError(f"Unknown initialization: {initialization}")
    if not np.isfinite(stochastic_strength) or stochastic_strength < 0:
        raise ValueError("stochastic_strength must be finite and nonnegative")
    bridge_strength = None
    black = None
    if initialization == "singular_black":
        if sampler != "stochastic":
            raise ValueError(
                "The singular bridge requires its matched stochastic sampler"
            )
        if getattr(candidate, "singular_path_version", None) != SINGULAR_PATH_VERSION:
            raise ValueError(
                "Train or load the new stochastic-bridge models before sampling"
            )
        bridge_strength = float(candidate.singular_bridge_strength)
        if not np.isfinite(bridge_strength) or bridge_strength <= 0:
            raise ValueError("The trained bridge strength must be finite and positive")
        black = binary_embeddings[0].to(initial_state).view(1, EMBEDDING_DIM, 1, 1)
        if not torch.equal(initial_state, black.expand_as(initial_state)):
            raise ValueError(
                "The singular bridge must start from the exact black embedding grid"
            )
    candidate.eval()
    state = initial_state.detach().clone()
    dt = 1.0 / steps
    capture_steps = set(capture_steps or ())
    trajectory = {0: state.detach().clone()} if 0 in capture_steps else {}
    for step in range(steps):
        t = step / steps
        times = torch.full(
            (state.shape[0],),
            t * TIME_EMBEDDING_SCALE,
            device=state.device,
            dtype=state.dtype,
        )
        prediction = candidate(state, times)
        if initialization == "singular_black":
            # Exact conditional bridge step with the inferred endpoint held fixed.
            u = (step + 1) / steps
            remaining = 1.0 - t
            next_remaining = 1.0 - u
            endpoint = singular_endpoint_from_drift(state, prediction, t, black)
            mean_t = remaining * black + t * endpoint
            mean_u = next_remaining * black + u * endpoint
            if next_remaining == 0.0:
                state = endpoint
            else:
                epsilon = torch.randn(
                    state.shape,
                    generator=generator,
                    device=state.device,
                    dtype=state.dtype,
                )
                state = (
                    mean_u
                    + np.sqrt(next_remaining / remaining) * (state - mean_t)
                    + np.sqrt(2.0 * bridge_strength * next_remaining * dt) * epsilon
                )
        elif sampler == "stochastic" and stochastic_strength > 0:
            epsilon = torch.randn(
                state.shape, generator=generator, device=state.device, dtype=state.dtype
            )
            drift = (
                1.0 + stochastic_strength * t
            ) * prediction - stochastic_strength * state
            noise = np.sqrt(2.0 * stochastic_strength * (1.0 - t) * dt) * epsilon
            state = state + dt * drift + noise
        else:
            state = state + dt * prediction
        completed_step = step + 1
        if completed_step in capture_steps:
            trajectory[completed_step] = state.detach().clone()
    if capture_steps:
        return state, trajectory
    return state


def benchmark_progress(description, total):
    progress = widgets.IntProgress(
        value=0,
        min=0,
        max=total,
        description=description,
        bar_style="info",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="100%"),
    )
    display(progress)
    return progress


if N_METRIC_SAMPLES < 1 or METRIC_BATCH_SIZE < 1:
    raise ValueError("Sample count and batch size must be positive")
if STOCHASTIC_STRENGTH < 0 or not np.isfinite(STOCHASTIC_STRENGTH):
    raise ValueError("STOCHASTIC_STRENGTH must be finite and nonnegative")
benchmark_config = {
    "steps": GENERATION_STEPS,
    "stochastic_strength": STOCHASTIC_STRENGTH,
    "singular_bridge_strength": SINGULAR_BRIDGE_STRENGTH,
    "singular_path_version": SINGULAR_PATH_VERSION,
    "samples_per_model": N_METRIC_SAMPLES,
    "batch_size": METRIC_BATCH_SIZE,
}
evaluation_initial_states_by_size = {}
solvability_results_by_size = {}
n_batches = int(np.ceil(N_METRIC_SAMPLES / METRIC_BATCH_SIZE))
n_runs = sum(
    sum(
        1
        for name in models
        if MODEL_META[name]["initialization"] == initialization
        and MODEL_META[name]["objective"] == objective
    )
    for initialization, objective, _sampler in BENCHMARK_GROUPS
)
evaluation_progress = benchmark_progress(
    "Generate benchmark samples", n_runs * len(EVALUATION_MAZE_SIZES) * n_batches
)
for maze_size in EVALUATION_MAZE_SIZES:
    evaluation_rng = torch.Generator(device=DEVICE).manual_seed(SEED + 501 + maze_size)
    initial_states = torch.randn(
        N_METRIC_SAMPLES,
        EMBEDDING_DIM,
        maze_size,
        maze_size,
        generator=evaluation_rng,
        device=DEVICE,
        dtype=binary_embeddings.dtype,
    )
    evaluation_initial_states_by_size[maze_size] = initial_states
    black_start = (
        binary_embeddings[0]
        .view(1, EMBEDDING_DIM, 1, 1)
        .expand(N_METRIC_SAMPLES, EMBEDDING_DIM, maze_size, maze_size)
        .contiguous()
    )
    results_at_size = {}
    for initialization, objective, sampler in BENCHMARK_GROUPS:
        candidates = [
            (name, candidate)
            for name, candidate in models.items()
            if MODEL_META[name]["initialization"] == initialization
            and MODEL_META[name]["objective"] == objective
        ]
        starts = initial_states if initialization == "gaussian" else black_start
        for name, candidate in candidates:
            generated_batches = []
            for batch_index, start in enumerate(
                range(0, N_METRIC_SAMPLES, METRIC_BATCH_SIZE)
            ):
                step_rng = torch.Generator(device=DEVICE).manual_seed(
                    SEED + 100_003 * maze_size + 1009 * batch_index
                )
                continuous = sample_flow(
                    candidate,
                    starts[start : start + METRIC_BATCH_SIZE],
                    steps=GENERATION_STEPS,
                    sampler=sampler,
                    stochastic_strength=STOCHASTIC_STRENGTH,
                    initialization=initialization,
                    generator=step_rng,
                )
                generated_batches.append(
                    torch.einsum("bdhw,kd->bkhw", continuous, binary_embeddings)
                    .argmax(dim=1)
                    .cpu()
                    .numpy()
                    .astype(np.uint8)
                )
                evaluation_progress.value += 1
            generated_mazes = np.concatenate(generated_batches, axis=0)
            solved = np.asarray(
                [solvability_rate([maze]) == 1.0 for maze in generated_mazes]
            )
            run_key = f"{name}/{sampler}"
            results_at_size[run_key] = {
                "model": name,
                "objective": objective,
                "replicate": MODEL_META[name]["replicate"],
                "initialization": initialization,
                "sampler": sampler,
                "bridge_strength": getattr(candidate, "singular_bridge_strength", None),
                "path_version": MODEL_META[name].get("path_version", "gaussian_linear"),
                "solvable": solved,
                "mazes": generated_mazes,
                "rate": float(solved.mean()),
            }
    solvability_results_by_size[maze_size] = results_at_size
evaluation_progress.bar_style = "success"

for maze_size in EVALUATION_MAZE_SIZES:
    results = solvability_results_by_size[maze_size]
    for group in BENCHMARK_GROUPS:
        initialization, objective, sampler = group
        group_results = [
            (key, result)
            for key, result in results.items()
            if result["initialization"] == initialization
            and result["objective"] == objective
            and result["sampler"] == sampler
        ]
        n_cols = 5
        n_rows = int(np.ceil(len(group_results) / n_cols))
        titles = [
            f"Run {result['replicate']:02d} | "
            f"{'solvable' if result['solvable'][0] else 'unsolvable'}"
            for _key, result in group_results
        ]
        examples_fig = make_subplots(
            rows=n_rows,
            cols=n_cols,
            subplot_titles=titles,
            horizontal_spacing=0.025,
            vertical_spacing=0.08,
        )
        for index, (_key, result) in enumerate(group_results):
            row, col = divmod(index, n_cols)
            examples_fig.add_trace(
                go.Heatmap(
                    z=result["mazes"][0],
                    zmin=0,
                    zmax=1,
                    colorscale=[[0, "black"], [1, "white"]],
                    showscale=False,
                ),
                row=row + 1,
                col=col + 1,
            )
            x_anchor = "x" if index == 0 else f"x{index + 1}"
            examples_fig.update_yaxes(
                scaleanchor=x_anchor,
                scaleratio=1,
                autorange="reversed",
                showticklabels=False,
                row=row + 1,
                col=col + 1,
            )
            examples_fig.update_xaxes(showticklabels=False, row=row + 1, col=col + 1)
        examples_fig.update_layout(
            title=f"{maze_size}x{maze_size} | {GENERATION_STEPS}-step samples | {BENCHMARK_LABELS[group]}",
            template="plotly_white",
            autosize=True,
            height=260 * n_rows + 90,
            margin=dict(l=20, r=20, t=90, b=20),
        )
        examples_fig.show(config={"responsive": True})

IntProgress(value=0, bar_style='info', description='Generate benchmark samples', layout=Layout(width='100%'), …

### 6.1 Example generation trajectories

For each of the six benchmark combinations, follow the first paired initialization from the same Gaussian sample or exact black grid. The six panels in a row show decoded states at steps 0, 20, 40, 60, 80 and 100. The stochastic rows use the same seeded Brownian increments; Gaussian-prior rows use the score-corrected SDE, while black-start rows use the stochastic bridge matched to their training path.

In [10]:
TRAJECTORY_STEPS = tuple(range(0, GENERATION_STEPS + 1, GENERATION_STEPS // 5))
trajectory_figures = {}
for maze_size in EVALUATION_MAZE_SIZES:
    rows = []
    for group in BENCHMARK_GROUPS:
        initialization, objective, sampler = group
        candidate_name = next(
            name
            for name in models
            if MODEL_META[name]["initialization"] == initialization
            and MODEL_META[name]["objective"] == objective
            and MODEL_META[name]["replicate"] == 1
        )
        if initialization == "gaussian":
            start_state = evaluation_initial_states_by_size[maze_size][:1]
        else:
            start_state = (
                binary_embeddings[0]
                .view(1, EMBEDDING_DIM, 1, 1)
                .expand(1, EMBEDDING_DIM, maze_size, maze_size)
                .contiguous()
            )
        path_rng = torch.Generator(device=DEVICE).manual_seed(
            SEED + 700_001 * maze_size
        )
        _final, path_states = sample_flow(
            models[candidate_name],
            start_state,
            steps=GENERATION_STEPS,
            sampler=sampler,
            stochastic_strength=STOCHASTIC_STRENGTH,
            initialization=initialization,
            generator=path_rng,
            capture_steps=TRAJECTORY_STEPS,
        )
        rows.append((group, path_states))

    trajectory_fig = make_subplots(
        rows=len(BENCHMARK_GROUPS),
        cols=len(TRAJECTORY_STEPS),
        subplot_titles=[
            f"Step {step}" for _group in BENCHMARK_GROUPS for step in TRAJECTORY_STEPS
        ],
        vertical_spacing=0.045,
        horizontal_spacing=0.012,
    )
    for row_index, (group, path_states) in enumerate(rows, start=1):
        for col_index, step in enumerate(TRAJECTORY_STEPS, start=1):
            continuous = path_states[step]
            pixel_probabilities = torch.einsum(
                "bdhw,kd->bkhw", continuous, binary_embeddings
            ).softmax(dim=1)
            decoded = pixel_probabilities.argmax(dim=1)[0].cpu().numpy()
            trajectory_fig.add_trace(
                go.Heatmap(
                    z=decoded,
                    zmin=0,
                    zmax=1,
                    colorscale=[[0, "black"], [1, "white"]],
                    showscale=False,
                ),
                row=row_index,
                col=col_index,
            )
            x_axis_index = (row_index - 1) * len(TRAJECTORY_STEPS) + col_index
            x_anchor = "x" if x_axis_index == 1 else f"x{x_axis_index}"
            trajectory_fig.update_yaxes(
                scaleanchor=x_anchor,
                scaleratio=1,
                autorange="reversed",
                showticklabels=False,
                title_text=BENCHMARK_TICKS[group] if col_index == 1 else None,
                row=row_index,
                col=col_index,
            )
            trajectory_fig.update_xaxes(
                showticklabels=False, row=row_index, col=col_index
            )
    trajectory_fig.update_layout(
        title=f"Sample trajectories | {maze_size}x{maze_size} | {GENERATION_STEPS} steps",
        template="plotly_white",
        autosize=True,
        height=170 * len(BENCHMARK_GROUPS) + 150,
        margin=dict(l=190, r=20, t=100, b=25),
    )
    trajectory_figures[maze_size] = trajectory_fig
    trajectory_fig.show(config={"responsive": True})

### 6.2 Solvability comparison

Each box summarizes the 10 trained initializations in its objective, sampler and initialization group. Gaussian groups share the same initial samples at each resolution. Black-bridge groups start from the same exact black grid and use the same seeded noise draws. Dark horizontal marks show group means.


In [11]:
def plot_benchmark_metric(
    values_by_size, metric, title, y_title, y_range=None, percent=False
):
    fig = make_subplots(
        rows=1,
        cols=len(EVALUATION_MAZE_SIZES),
        subplot_titles=[f"{size}x{size}" for size in EVALUATION_MAZE_SIZES],
        horizontal_spacing=0.10,
    )
    for col, maze_size in enumerate(EVALUATION_MAZE_SIZES, start=1):
        results = solvability_results_by_size[maze_size]
        for group_index, group in enumerate(BENCHMARK_GROUPS):
            initialization, objective, sampler = group
            keys = [
                key
                for key, result in results.items()
                if result["initialization"] == initialization
                and result["objective"] == objective
                and result["sampler"] == sampler
                and np.isfinite(values_by_size[maze_size][key][metric])
            ]
            values = np.asarray(
                [values_by_size[maze_size][key][metric] for key in keys]
            )
            if not len(values):
                continue
            color = BENCHMARK_COLORS[group]
            fig.add_trace(
                go.Box(
                    x=[group_index] * len(values),
                    y=values,
                    name=BENCHMARK_LABELS[group],
                    legendgroup=BENCHMARK_LABELS[group],
                    boxpoints=False,
                    marker=dict(color=color, size=7, opacity=0.78),
                    line=dict(color=color, width=2),
                    customdata=np.asarray(keys)[:, None],
                    hovertemplate="%{customdata[0]}<br>Model mean: %{y:.2f}<extra></extra>",
                    showlegend=(col == 1),
                ),
                row=1,
                col=col,
            )
            mean_value = float(values.mean())
            fig.add_shape(
                type="line",
                x0=group_index - 0.2,
                x1=group_index + 0.2,
                y0=mean_value,
                y1=mean_value,
                line=dict(color="#111827", width=3),
                row=1,
                col=col,
            )
            suffix = "%" if percent else ""
            fig.add_annotation(
                x=group_index,
                y=mean_value,
                text=f"mean {mean_value:.2f}{suffix}",
                showarrow=False,
                yshift=15,
                font=dict(size=9, color="#111827"),
                bgcolor="rgba(255,255,255,0.86)",
                row=1,
                col=col,
            )
        fig.update_xaxes(
            title_text="Initialization / objective / sampler",
            tickmode="array",
            tickvals=list(range(len(BENCHMARK_GROUPS))),
            ticktext=[BENCHMARK_TICKS[group] for group in BENCHMARK_GROUPS],
            range=[-0.5, len(BENCHMARK_GROUPS) - 0.5],
            tickfont=dict(size=10),
            row=1,
            col=col,
        )
        fig.update_yaxes(
            title_text=y_title, range=y_range, rangemode="tozero", row=1, col=col
        )
    fig.update_layout(
        title=(
            f"{title} | {benchmark_config['steps']} steps | "
            f"Gaussian noise={benchmark_config['stochastic_strength']:g}, "
            f"bridge noise={benchmark_config['singular_bridge_strength']:g}"
        ),
        template="plotly_white",
        autosize=True,
        height=650,
        margin=dict(l=75, r=30, t=110, b=190),
        legend=dict(orientation="h", y=-0.38, x=0),
    )
    fig.show(config={"responsive": True})
    return fig


solvability_values_by_size = {
    size: {
        key: {"solvability_percent": 100.0 * result["rate"]}
        for key, result in results.items()
    }
    for size, results in solvability_results_by_size.items()
}
solvability_box = plot_benchmark_metric(
    solvability_values_by_size,
    "solvability_percent",
    "Solvability across weight initializations",
    "Solvability (%)",
    y_range=[0, 100],
    percent=True,
)

## 7. Solution length and A* search effort

For each cached maze at both resolutions and for all six initialization/objective/sampler combinations, calculate shortest entrance-to-exit path length with BFS and count A* expansions until the exit is reached. The goal is included in the expansion count. Unsolvable mazes have undefined path length; A* expansions count all explored cells before the frontier is exhausted.

Plots compare per-model means across generated mazes. Shortest-length means use solvable mazes only; models with no solved samples are omitted. A* means include every generated maze. Horizontal marks show each group mean across models.

In [12]:
from collections import deque
import heapq

MAZE_DIRECTIONS = ((-1, 0), (1, 0), (0, -1), (0, 1))


def shortest_path_length_and_astar_expansions(maze):
    passable = np.asarray(maze, dtype=bool)
    height, width = passable.shape
    start, goal = (0, 1), (height - 1, width - 2)

    def neighbors(cell):
        row, col = cell
        for dr, dc in MAZE_DIRECTIONS:
            nr, nc = row + dr, col + dc
            if 0 <= nr < height and 0 <= nc < width and passable[nr, nc]:
                yield nr, nc

    shortest_length = np.nan
    if passable[start]:
        distances = {start: 0}
        queue = deque([start])
        while queue:
            cell = queue.popleft()
            for nxt in neighbors(cell):
                if nxt not in distances:
                    distances[nxt] = distances[cell] + 1
                    queue.append(nxt)
        if goal in distances:
            shortest_length = float(distances[goal])

    if not passable[start]:
        return shortest_length, 0

    def heuristic(cell):
        return abs(cell[0] - goal[0]) + abs(cell[1] - goal[1])

    frontier = [(heuristic(start), 0, start)]
    best_cost = {start: 0}
    closed = set()
    expansions = 0
    while frontier:
        _, cost, cell = heapq.heappop(frontier)
        if cell in closed or cost != best_cost.get(cell):
            continue
        closed.add(cell)
        expansions += 1
        if cell == goal:
            break
        for nxt in neighbors(cell):
            new_cost = cost + 1
            if new_cost < best_cost.get(nxt, float("inf")):
                best_cost[nxt] = new_cost
                heapq.heappush(frontier, (new_cost + heuristic(nxt), new_cost, nxt))
    return shortest_length, expansions


maze_metrics_by_size = {}
search_progress = benchmark_progress(
    "Search metrics",
    sum(len(results) for results in solvability_results_by_size.values()),
)
for maze_size in EVALUATION_MAZE_SIZES:
    maze_metrics_by_size[maze_size] = {}
    for run_key, result in solvability_results_by_size[maze_size].items():
        per_maze = [
            shortest_path_length_and_astar_expansions(maze) for maze in result["mazes"]
        ]
        lengths = np.asarray([item[0] for item in per_maze], dtype=float)
        expansions = np.asarray([item[1] for item in per_maze], dtype=float)
        maze_metrics_by_size[maze_size][run_key] = {
            "shortest_solution_length": (
                float(np.nanmean(lengths)) if np.isfinite(lengths).any() else np.nan
            ),
            "astar_expansions": float(expansions.mean()),
        }
        search_progress.value += 1
search_progress.bar_style = "success"

shortest_length_fig = plot_benchmark_metric(
    maze_metrics_by_size,
    "shortest_solution_length",
    "Shortest solution length",
    "Grid moves",
)
astar_fig = plot_benchmark_metric(
    maze_metrics_by_size,
    "astar_expansions",
    "A* search effort until exit",
    "Expanded cells",
)

IntProgress(value=0, bar_style='info', description='Search metrics', layout=Layout(width='100%'), max=120, sty…

## 8. Maze-space accessibility

For every cached maze at both resolutions and for all six initialization/objective/sampler combinations, measure the fraction of white passage cells reachable from the entrance and count white connected components that exclude the entrance. Each box shows per-model means across all generated samples, including unsolvable mazes; horizontal marks show group means.

In [13]:
OPTIMIZATION_START = (0, 1)
OPTIMIZATION_DIRECTIONS = ((-1, 0), (1, 0), (0, -1), (0, 1))


def maze_accessibility_metrics(maze):
    white = np.asarray(maze) == 1
    height, width = white.shape

    def neighbors(cell):
        row, col = cell
        for dr, dc in OPTIMIZATION_DIRECTIONS:
            nr, nc = row + dr, col + dc
            if 0 <= nr < height and 0 <= nc < width and white[nr, nc]:
                yield nr, nc

    white_cells = {(int(r), int(c)) for r, c in np.argwhere(white)}
    reachable = set()
    if OPTIMIZATION_START in white_cells:
        reachable.add(OPTIMIZATION_START)
        queue = [OPTIMIZATION_START]
        while queue:
            cell = queue.pop()
            for nxt in neighbors(cell):
                if nxt not in reachable:
                    reachable.add(nxt)
                    queue.append(nxt)

    unseen = set(white_cells)
    isolated_components = 0
    while unseen:
        seed = unseen.pop()
        component = {seed}
        queue = [seed]
        while queue:
            cell = queue.pop()
            for nxt in neighbors(cell):
                if nxt in unseen:
                    unseen.remove(nxt)
                    component.add(nxt)
                    queue.append(nxt)
        if not reachable or not (component & reachable):
            isolated_components += 1

    reachable_white_percent = (
        100.0 * len(reachable) / len(white_cells) if white_cells else 0.0
    )
    return reachable_white_percent, isolated_components


accessibility_by_size = {}
accessibility_progress = benchmark_progress(
    "Accessibility",
    sum(len(results) for results in solvability_results_by_size.values()),
)
for maze_size in EVALUATION_MAZE_SIZES:
    accessibility_by_size[maze_size] = {}
    for run_key, result in solvability_results_by_size[maze_size].items():
        per_maze = [maze_accessibility_metrics(maze) for maze in result["mazes"]]
        accessibility_by_size[maze_size][run_key] = {
            "reachable_white_percent": float(np.mean([item[0] for item in per_maze])),
            "isolated_components": float(np.mean([item[1] for item in per_maze])),
        }
        accessibility_progress.value += 1
accessibility_progress.bar_style = "success"

reachable_fig = plot_benchmark_metric(
    accessibility_by_size,
    "reachable_white_percent",
    "White cells reachable from entrance",
    "Reachable white cells (%)",
    y_range=[0, 100],
    percent=True,
)
isolated_fig = plot_benchmark_metric(
    accessibility_by_size,
    "isolated_components",
    "White components isolated from entrance",
    "Isolated component count",
)

IntProgress(value=0, bar_style='info', description='Accessibility', layout=Layout(width='100%'), max=120, styl…